# Notebook 3 — QA Checks, Thresholds & Data Leakage

**Webinar 1: The AI-Ready Data Audit**

---

### Purpose

Score the dataset across four quality dimensions — **Completeness, Validity,
Uniqueness, and Consistency** — using explicit boolean checks that flag
every defective row.

### Input / Output

| | File |
|---|---|
| Input | `data/profiled_data.xlsx` |
| Output | `data/qa_checked_data.xlsx` |

In [ ]:
import os
import pandas as pd
import numpy as np

DATA_DIR = os.path.join("..", "data")
input_file = os.path.join(DATA_DIR, "profiled_data.xlsx")

df = pd.read_excel(input_file, sheet_name="raw_data", engine="openpyxl")

print("=" * 60)
print("  NOTEBOOK 3 — QA CHECKS, THRESHOLDS & DATA LEAKAGE")
print("=" * 60)
print(f"\n✅ Loaded: {input_file}")
print(f"   Shape : {df.shape[0]} rows × {df.shape[1]} columns")

---

## 1. COMPLETENESS — Missing Values

In [ ]:
# ===========================================================================
# Completeness: fraction of values that are NOT missing
# ===========================================================================
completeness_per_col = (1 - df.isnull().mean()) * 100
overall_completeness = round(completeness_per_col.mean(), 1)

print("=" * 55)
print("1. COMPLETENESS  (% non-missing)")
print("=" * 55)
print(completeness_per_col.round(1).to_string())
print(f"\n   Overall completeness: {overall_completeness}%")

---

## 2. VALIDITY — Boolean Threshold Checks

We write explicit boolean masks to flag rows with impossible values.
These are not statistical outliers — they are **hard violations** of
business rules (marks can't be negative, attendance can't exceed 100%).

In [ ]:
# ===========================================================================
# Validity: explicit boolean checks per business rule
# ===========================================================================
validity_rules = {
    "marks_math":     (0, 100),
    "marks_science":  (0, 100),
    "attendance_pct": (0, 100),
}

print("=" * 55)
print("2. VALIDITY  (values within allowed range)")
print("=" * 55)

validity_issues = []
for col, (lo, hi) in validity_rules.items():
    valid_mask = df[col].notna()
    # Boolean mask: True where the value violates the rule
    too_low = df.loc[valid_mask, col] < lo
    too_high = df.loc[valid_mask, col] > hi
    bad_count = too_low.sum() + too_high.sum()
    total = valid_mask.sum()
    pct_valid = round((1 - bad_count / total) * 100, 1) if total else 100.0

    validity_issues.append({
        "column": col,
        "rule": f"{lo} <= value <= {hi}",
        "checked": total,
        "invalid": int(bad_count),
        "validity_pct": pct_valid,
    })

    # Print the actual offending rows
    offenders = df.loc[valid_mask & (too_low | too_high), ["student_id", col]]
    if not offenders.empty:
        print(f"\n   ⚠️  {col} — {bad_count} violation(s):")
        print(f"   {offenders.to_string(index=False)}")

validity_df = pd.DataFrame(validity_issues)
overall_validity = round(validity_df["validity_pct"].mean(), 1)

print(f"\n{validity_df.to_string(index=False)}")
print(f"\n   Overall validity: {overall_validity}%")

---

## 3. UNIQUENESS — Duplicate Detection

### 🏛️ Architectural Deep-Dive: Data Leakage from Duplicates

**Data Leakage** is what happens when information from outside the training
set bleeds into the model evaluation. The most common (and most invisible)
cause: **exact duplicate rows** that survive into a Train/Test split.

If Student #8 appears in both the Training set and the Test set, the model
has literally *memorized* that student's outcome during training. When it
encounters the same row during evaluation, it "predicts" perfectly — not
because it learned a pattern, but because it is recalling a memorized answer.

The result: **inflated accuracy metrics** that collapse the moment the model
sees genuinely new data in production. This is the **Accuracy Paradox** — the
model looks perfect in staging and fails in production.

> Failing to drop exact duplicates before a Train/Test split is one of the
> most expensive mistakes in the ML lifecycle because it is invisible
> until deployment.

In [ ]:
# ===========================================================================
# Uniqueness: identify exact-duplicate rows
# ===========================================================================
full_dup_mask = df.duplicated(keep=False)  # keep=False flags ALL copies
full_dup_count = df.duplicated().sum()     # keep='first' for count
id_dup_count = df["student_id"].duplicated().sum()
uniqueness_score = round((1 - full_dup_count / len(df)) * 100, 1)

print("=" * 55)
print("3. UNIQUENESS  (no unwarranted duplicates)")
print("=" * 55)
print(f"   Fully duplicate rows   : {full_dup_count}")
print(f"   Duplicate student_ids  : {id_dup_count}")
print(f"   Uniqueness score       : {uniqueness_score}%")

if full_dup_count > 0:
    print(f"\n   ⚠️  Duplicate rows (all copies shown):")
    dup_rows = df[full_dup_mask].sort_values("student_id")
    print(dup_rows[["student_id", "marks_math", "marks_science", "attendance_pct", "city"]].to_string(index=True))
    print(
        "\n   🚨 DATA LEAKAGE RISK: These identical rows would leak into both"
        "\n   Train and Test partitions, inflating validation accuracy."
        "\n   They will be dropped in Notebook 4."
    )

---

## 4. CONSISTENCY — Formatting vs. Entity Resolution

### 🏛️ Architectural Deep-Dive: Consistency ≠ Uniqueness

These two quality dimensions are frequently confused. They are architecturally
distinct problems:

| Dimension | Problem Type | Example | Fix |
|-----------|-------------|---------|-----|
| **Consistency** | FORMATTING — same entity, different encoding | 'Bangalore' vs 'Bengaluru' vs 'bangalore ' | Canonical mapping (controlled vocabulary) |
| **Uniqueness** | ENTITY RESOLUTION — same event, multiple records | Row 8 appears twice in the dataset | `drop_duplicates()` |

### Why Fix Consistency BEFORE Uniqueness

Normalising labels first may surface duplicates that were hidden by formatting
mismatches. Two rows that looked different because one said 'Bangalore' and the
other said 'Bengaluru' may be revealed as duplicates once both are mapped to
the canonical 'Bengaluru'.

In enterprise systems, the canonical mapping lives in a **Master Data Management
(MDM)** layer — not in ad-hoc string replacements scattered across notebooks.

In [ ]:
# ===========================================================================
# Consistency: analyse the city column
# ===========================================================================
city_raw = df["city"].dropna()
city_normalised = city_raw.str.strip().str.title()

distinct_raw = city_raw.nunique()
distinct_normalised = city_normalised.nunique()
inconsistency_count = distinct_raw - distinct_normalised
consistency_score = round((distinct_normalised / distinct_raw) * 100, 1) if distinct_raw else 100.0

# Build mapping table
consistency_map = (
    pd.DataFrame({"raw": city_raw, "normalised": city_normalised})
    .drop_duplicates()
    .sort_values("normalised")
    .reset_index(drop=True)
)

print("=" * 55)
print("4. CONSISTENCY  (uniform representation)")
print("=" * 55)
print(f"   Column checked         : city")
print(f"   Distinct raw values    : {distinct_raw}")
print(f"   After normalisation    : {distinct_normalised}")
print(f"   Inconsistent spellings : {inconsistency_count}")
print(f"   Consistency score      : {consistency_score}%")
print(f"\n   Raw → Normalised mapping:")
print(consistency_map.to_string(index=False))

print(
    "\n   📌 Note: 'Bangalore' and 'Bengaluru' are the SAME city."
    "\n   Simple title-casing won't catch this — we need a canonical"
    "\n   mapping dictionary (applied in Notebook 4)."
)

In [ ]:
# ===========================================================================
# Overall Quality Scorecard
# ===========================================================================
scorecard = pd.DataFrame([
    {"dimension": "Completeness", "score_pct": overall_completeness,
     "description": "Fraction of non-missing values"},
    {"dimension": "Validity", "score_pct": overall_validity,
     "description": "Values within allowed ranges"},
    {"dimension": "Uniqueness", "score_pct": uniqueness_score,
     "description": "No unwarranted duplicate rows"},
    {"dimension": "Consistency", "score_pct": consistency_score,
     "description": "Uniform representation of same entity"},
])

overall_quality = round(scorecard["score_pct"].mean(), 1)

print(f"\n{'=' * 55}")
print("OVERALL QUALITY SCORECARD")
print("=" * 55)
print(scorecard.to_string(index=False))
print(f"\n   Overall quality score: {overall_quality}%")

In [ ]:
# ===========================================================================
# Export — raw data + scorecard + validity detail + consistency map
# ===========================================================================
output_file = os.path.join(DATA_DIR, "qa_checked_data.xlsx")

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="raw_data", index=False)
    scorecard.to_excel(writer, sheet_name="scorecard", index=False)
    validity_df.to_excel(writer, sheet_name="validity_detail", index=False)
    consistency_map.to_excel(writer, sheet_name="consistency_map", index=False)

print(f"\n✅ Saved: {output_file}")
print(f"   Sheets: raw_data, scorecard, validity_detail, consistency_map")
print(f"\n→ Next: Notebook 4 — Rescue, Reject & KNN Imputation")